<a href="https://colab.research.google.com/github/2303A52054/HPC/blob/main/HPC_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numba numpy
import numpy as np
from numba import njit, prange
import time

In [2]:
#Assignment 1
N = 10_000_000
A = np.random.rand(N)
B = np.random.rand(N)
C1 = np.zeros(N)
C2 = np.zeros(N)

# Normal Loop (Scalar)
start = time.time()
for i in range(N):
    C1[i] = A[i] + B[i]
print("Normal Loop Time:", time.time() - start)

# SIMD-like Loop with Numba
@njit(parallel=True, fastmath=True)
def simd_add(A, B):
    C = np.empty_like(A)
    for i in prange(A.size):  # Parallel loop (SIMD-style)
        C[i] = A[i] + B[i]
    return C

start = time.time()
C2 = simd_add(A, B)
print("SIMD (Numba) Time:", time.time() - start)

print("Arrays Equal:", np.allclose(C1, C2))

Normal Loop Time: 2.8361568450927734
SIMD (Numba) Time: 1.399662971496582
Arrays Equal: True


In [3]:
#Assignment 2
arr = np.random.rand(N)

# Normal Sum
start = time.time()
s1 = 0.0
for i in range(N):
    s1 += arr[i]
print("Normal Loop Time:", time.time() - start)

# SIMD Reduction with Numba
@njit(parallel=True, fastmath=True)
def simd_sum(arr):
    total = 0.0
    for i in prange(arr.size):
        total += arr[i]
    return total

start = time.time()
s2 = simd_sum(arr)
print("SIMD Reduction Time:", time.time() - start)

print("Equal:", np.isclose(s1, s2))

Normal Loop Time: 4.5604236125946045
SIMD Reduction Time: 0.5272378921508789
Equal: True


In [4]:
#Assignment 3
A_aligned = np.random.rand(N)
B_aligned = np.random.rand(N)

# Misaligned arrays (simulate by offset)
A_unaligned = A_aligned[1:]
B_unaligned = B_aligned[1:]

@njit(parallel=True, fastmath=True)
def simd_mul(A, B):
    C = np.empty_like(A)
    for i in prange(A.size):
        C[i] = A[i] * B[i]
    return C

start = time.time()
simd_mul(A_aligned, B_aligned)
print("Aligned Time:", time.time() - start)

start = time.time()
simd_mul(A_unaligned, B_unaligned)
print("Unaligned Time:", time.time() - start)

Aligned Time: 0.29438138008117676
Unaligned Time: 0.02678656578063965


In [5]:
#Assignment 4
@njit(parallel=True, fastmath=True)
def parallel_simd(A, B):
    C = np.empty_like(A)
    for i in prange(A.size):  # Parallel + SIMD
        C[i] = A[i] * B[i] + np.sin(A[i])
    return C

start = time.time()
C = parallel_simd(A, B)
print("Parallel + SIMD Time:", time.time() - start)

Parallel + SIMD Time: 0.35086989402770996


In [6]:
#Assignment 5
@njit(parallel=True)
def conditional_loop(A, B):
    C = np.empty_like(A)
    for i in prange(A.size):
        if A[i] > 0.5:
            C[i] = A[i] * B[i]
        else:
            C[i] = A[i] - B[i]
    return C

@njit(parallel=True, fastmath=True)
def simd_friendly(A, B):
    C = A * B * (A > 0.5) + (A - B) * (A <= 0.5)
    return C

start = time.time()
conditional_loop(A, B)
print("Branch Loop Time:", time.time() - start)

start = time.time()
simd_friendly(A, B)
print("SIMD-Friendly Vectorized Time:", time.time() - start)

Branch Loop Time: 0.3453221321105957
SIMD-Friendly Vectorized Time: 0.47469353675842285
